This colab was created originally by daswer123 for
https://github.com/minipasila/echo-tts-api-server

**Updated for Echo TTS**

Maybe you wanted to try Echo TTS but your computer doesn't allow it, then this colab will help you.

---

## About Echo TTS

Echo TTS is a 2.4B parameter diffusion model that provides:
- **High Quality:** 44.1kHz audio output
- **Fast Generation:** ~0.5s for 10s text
- **Voice Cloning:** Supports reference audio up to 5 minutes
- **Hardware:** Requires 8GB+ VRAM (Colab T4 has 16GB)
- **Language:** Primarily English (limited multilingual support)

**Model Size:** ~5GB total download
**Max Duration:** 30 seconds per request

# INSTALL (takes 3-5 minutes)

In [ ]:
#@markdown #####You can enable gdrive that drag and drop samples directly through your google drive via the `drive/Mydrive` path
mount_gdrive = False #@param{"type":"boolean"}

if mount_gdrive:
  from google.colab import drive
  drive.mount('/content/drive')

print("Installing Packages and Dependencies\nPlease wait 3-5 minutes")

# Install cloudflared for public access (more reliable than localtunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > '/dev/null' 2>&1

# Install PyTorch with CUDA 12.1 support (required for Echo TTS)
!pip -q install torch>=2.9.1 torchvision>=0.18.0 torchaudio>=2.9.1 --index-url https://download.pytorch.org/whl/cu121 > '/dev/null' 2>&1

# Install PortAudio development headers (required for PyAudio)
!apt-get update -qq
!apt-get install -y -qq portaudio19-dev python3-dev libportaudio2 libasound2-dev > '/dev/null' 2>&1

# Install Echo TTS library from GitHub
!pip -q install git+https://github.com/jordandare/echo-tts.git > '/dev/null' 2>&1

# Install echo-tts-api-server from repository
!git clone https://github.com/minipasila/echo-tts-api-server
%cd echo-tts-api-server
!pip -q install -e . > '/dev/null' 2>&1

# Install numpy (compatible version)
!pip -q install --force-reinstall numpy==1.26.2 > '/dev/null' 2>&1

print("Downloading sample speaker files")
# Create speakers directory
!mkdir -p speakers

# Download sample files from repository
!wget -q -O speakers/male.wav https://github.com/minipasila/echo-tts-api-server/raw/main/example/male.wav
!wget -q -O speakers/female.wav https://github.com/minipasila/echo-tts-api-server/raw/main/example/female.wav
!wget -q -O speakers/calm_female.wav https://github.com/minipasila/echo-tts-api-server/raw/main/example/calm_female.wav

print("Done!")
print("\nNote: Echo TTS models (~5GB) will be downloaded on first launch")

# Usage Tips

1. **First Run:** Echo TTS models will download automatically (~5GB, 1-2 minutes)
2. **Language:** Use English for best results (Echo TTS is primarily optimized for English)
3. **Speakers:** Use the provided sample files or upload your own to the `speakers/` folder
4. **Duration:** Keep text under 30 seconds for best results
5. **Quality:** Use clean, noise-free reference audio for voice cloning
6. **Hardware:** Colab T4 GPU (16GB VRAM) is sufficient for Echo TTS

# Launch

In [ ]:
#@markdown A link and ip address will appear in front of you. You need to follow the link and paste there the given ip address.

#@markdown After that you can copy the link and paste it into the tavern.

#@markdown Also if you are running for the first time you need to confirm the model download by typing "y" in the console

#@markdown **Important:** Echo TTS requires ~5GB of model downloads and 8GB+ VRAM. The first run will take 1-2 minutes to download models from HuggingFace.

#@markdown **Note:** Echo TTS is primarily optimized for English. Other languages may work but with lower quality.
import time
import subprocess
import threading

# Start cloudflared tunnel in background
print("Starting Cloudflare tunnel...")

# Function to run cloudflared and capture output
def run_tunnel():
    process = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8020'],
                                  stdout=subprocess.PIPE,
                                  stderr=subprocess.PIPE,
                                  text=True)
    for line in process.stdout:
        if 'https://' in line:
            print(f"\n{'='*60}")
            print(f"TUNNEL URL: {line.strip()}")
            print(f"{'='*60}")
            print("\nCopy the URL above and paste it into SillyTavern")
            print("\nStarting Echo TTS server...")
            print(f"{'='*60}\n")

# Start tunnel in background thread
tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)
tunnel_thread.start()

time.sleep(3)
print("Tunnel is starting... Please wait for the URL to appear above.")
print("\nYour public IP:")
!curl ipv4.icanhazip.com

# Start the server
!python -m xtts_api_server -hs 0.0.0.0